# Toistuvat neuroverkot

Edellisessä moduulissa käsittelimme rikkaiden semanttisten tekstiesitysten muodostamista. Käyttämämme arkkitehtuuri tallentaa sanojen kokoelman merkityksen lauseessa, mutta se ei ota huomioon sanojen **järjestystä**, koska upotusten jälkeinen yhdistämisoperaatio poistaa tämän tiedon alkuperäisestä tekstistä. Koska nämä mallit eivät pysty esittämään sanajärjestystä, ne eivät voi ratkaista monimutkaisempia tai monitulkintaisempia tehtäviä, kuten tekstin generaatiota tai kysymyksiin vastaamista.

Tallentaaksemme tekstijakson merkityksen käytämme neuroverkkorakennetta, jota kutsutaan **toistuvaksi neuroverkoksi** eli RNN:ksi. Käytettäessä RNN:ää syötämme lauseemme verkon läpi yksi token kerrallaan, ja verkko tuottaa jonkinlaisen **tilan**, jonka annamme verkolle uudelleen seuraavan tokenin kanssa.

![Kuva, joka näyttää esimerkin toistuvan neuroverkon generaatiosta.](../../../../../translated_images/fi/rnn.27f5c29c53d727b5.webp)

Annetusta token-jaksosta $X_0,\dots,X_n$ RNN luo sarjan neuroverkkolohkoja ja opettaa tämän sarjan päästä päähän takaisinkytkentämenetelmällä. Jokainen verkkolohko saa syötteenä parin $(X_i,S_i)$ ja tuottaa tulokseksi $S_{i+1}$. Lopputila $S_n$ tai ulostulo $Y_n$ syötetään lineaariselle luokittelijalle tuloksen tuottamiseksi. Kaikki verkkolohkot käyttävät samoja painoja, ja niitä opetetaan päästä päähän yhdellä takaisinkytkentäkierroksella.

> Kuvassa ylempänä toistuva neuroverkko on esitetty auki kirjoitettuna (vasemmalla) ja tiiviimmässä toistuvassa muodossa (oikealla). On tärkeää ymmärtää, että kaikilla RNN-solmuilla on samat **jaettavat painot**.

Koska tilavektorit $S_0,\dots,S_n$ kulkevat verkon läpi, RNN pystyy oppimaan sanojen välisiä sekvenssidependenssejä. Esimerkiksi, kun sana *not* esiintyy jossain jaksossa, se voi oppia kumoamaan tiettyjä osia tilavektorissa.

Sisäisesti jokainen RNN-solu sisältää kaksi painomatriisia: $W_H$ ja $W_I$, sekä vinouman $b$. Jokaisella RNN-askeleella, annettuna syöte $X_i$ ja syötetila $S_i$, ulostulotila lasketaan kaavalla $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, missä $f$ on aktivointifunktio (usein $\tanh$).

> Tekstintuotannon kaltaisissa tehtävissä (joista käsittelemme seuraavassa osassa) tai konekäännöksessä haluamme saada myös ulostulon jokaisella RNN-askeleella. Tässä tapauksessa on myös toinen matriisi $W_O$, ja ulostulo lasketaan kaavalla $Y_i=f(W_O\times S_i+b_O)$.

Katsotaanpa, miten toistuvat neuroverkot voivat auttaa meitä luokittelemaan uutisdatan.

> Sandbox-ympäristössä tarvitsemme seuraavan solun suorittamista varmistaaksemme tarvittavan kirjaston asennuksen ja datan esilatauksen. Jos ajat koodin paikallisesti, voit ohittaa seuraavan solun.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Suuria malleja koulutettaessa GPU-muistin allokaatio saattaa muodostua ongelmaksi. Saatamme myös joutua kokeilemaan eri minibatch-kokoja, jotta data mahtuu GPU-muistiimme, mutta koulutus on silti tarpeeksi nopeaa. Jos ajat tätä koodia omalla GPU-koneellasi, voit kokeilla säätää minibatch-kokoa koulutuksen nopeuttamiseksi.

> **Huom**: Tietyt NVidian ajuriversiot tunnetaan siitä, etteivät ne vapauta muistia mallin koulutuksen jälkeen. Suoritamme tässä muistikirjassa useita esimerkkejä, ja se saattaa aiheuttaa muistin loppumista tietyissä asetuksissa, erityisesti jos teet omia kokeilujasi saman muistikirjan osana. Jos kohtaat outoja virheitä koulutuksen alkaessa, saatat haluta käynnistää muistikirjan ytimen uudelleen.


In [3]:
batch_size = 16
embed_size = 64

## Yksinkertainen RNN-luokittelija

Yksinkertaisen RNN:n tapauksessa kukin toistuva yksikkö on yksinkertainen lineaarinen verkko, joka ottaa syötteenä sisääntulovektorin ja tilavektorin, ja tuottaa uuden tilavektorin. Kerasissa tämä voidaan esittää kerroksella `SimpleRNN`.

Vaikka voimme syöttää RNN-kerrokselle suoraan one-hot-koodattuja tokeneita, se ei ole hyvä idea niiden korkean ulottuvuuden vuoksi. Siksi käytämme upotuskerrosta sanavektorien ulottuvuuden alentamiseen, jota seuraa RNN-kerros ja lopuksi `Dense`-luokitin.

> **Huomautus**: Tapauksissa, joissa ulottuvuus ei ole kovin korkea, esimerkiksi kun käytetään merkkipohjaista tokenisointia, voi olla järkevää syöttää RNN-solulle suoraan one-hot-koodatut tokenit.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Huom:** Käytämme tässä yksinkertaisuuden vuoksi kouluttamatonta upotekerrosta, mutta parempien tulosten saavuttamiseksi voimme käyttää Word2Vec:iä hyödyntävää valmiiksi koulutettua upotekerrosta, kuten aiemmassa jaksossa kuvattiin. Olisi hyvä harjoitus mukauttaa tämä koodi toimimaan valmiiksi koulutettujen upotusten kanssa.

Nyt koulutetaan RNN. Yleisesti ottaen RNN:n kouluttaminen on melko vaikeaa, koska kun RNN-solut puretaan sekvenssin pituuden mukaan, taaksepäin tavoistuksessa mukana olevien kerrosten määrä on melko suuri. Siksi tarvitsemme pienemmän oppimisnopeuden ja verkon kouluttamisen suuremmalla datamäärällä tuottaaksemme hyviä tuloksia. Tämä voi kestää melko kauan, joten GPU:n käyttö on suositeltavaa.

Nopeuttaaksemme koulutusta, koulutamme RNN-mallin pelkästään uutisotsikoilla jättäen kuvauksen pois. Voit kokeilla koulutusta myös kuvauksen kanssa ja katsoa, onnistuuko mallin koulutus.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Huomaa**, että tarkkuus on täällä todennäköisesti alhaisempi, koska koulutamme vain uutisotsikoilla.


## Muuttuvapituisten jaksosten uudelleentarkastelu 

Muista, että `TextVectorization`-kerros täyttää automaattisesti muuttuvan pituiset jaksot minibatchissa täyte-tunnuksilla. Käy ilmi, että nämä tunnukset osallistuvat myös koulutukseen, ja ne voivat vaikeuttaa mallin konvergenssia.

On useita tapoja, joilla voimme minimoida täytteen määrää. Yksi niistä on järjestää aineisto uudelleen jakson pituuden mukaan ja ryhmitellä kaikki jaksot koon mukaan. Tämä voidaan tehdä käyttämällä `tf.data.experimental.bucket_by_sequence_length`-funktiota (katso [dokumentaatio](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Toinen tapa on käyttää **maskia**. Kerasissa jotkut kerrokset tukevat lisätuloa, joka näyttää, mitkä tunnukset otetaan huomioon koulutuksessa. Sisällyttääksemme maskauksen malliin, voimme joko lisätä erillisen `Masking`-kerroksen ([dokumentaatio](https://keras.io/api/layers/core_layers/masking/)), tai voimme määrittää `mask_zero=True`-parametrin `Embedding`-kerroksellemme.

> **Huom**: Tämä koulutus kestää noin 5 minuuttia yhden epookin suorittamiseen koko aineistolla. Voit keskeyttää koulutuksen milloin tahansa, jos kärsivällisyys loppuu. Voit myös rajata koulutuksessa käytettävän datan määrää lisäämällä `.take(...)`-lauseen `ds_train` ja `ds_test`-aineistojen jälkeen.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nyt kun käytämme maskausta, voimme kouluttaa mallin koko otsikoiden ja kuvauksien aineistolla.

> **Huomautus**: Oletko huomannut, että olemme käyttäneet uutisotsikoilla koulutettua vektorisointia, emme koko artikkelin tekstiä? Tämä voi mahdollisesti aiheuttaa joidenkin tokenien ohittamisen, joten vektorisoinnin uudelleenkoulutus olisi parempi. Se saattaa kuitenkin vaikuttaa vain hyvin vähän, joten pysymme aiemmin esikoulutetussa vektorisoinnissa yksinkertaisuuden vuoksi.


## LSTM: Pitkä lyhytkestoinen muisti

Yksi RNN:ien tärkeimmistä ongelmista on **häviävät gradientit**. RNN:it voivat olla melko pitkiä, ja gradienien kuljettaminen takaisin verkon ensimmäiselle kerrokselle takaisinkytkennän aikana voi olla vaikeaa. Kun näin tapahtuu, verkko ei pysty oppimaan yhteyksiä kaukana toisistaan olevien tokenien välillä. Yksi tapa välttää tämä ongelma on tuoda käyttöön **eksplisiittinen tilanhallinta** käyttämällä **portteja**. Kaksi yleisintä portteja käyttävää arkkitehtuuria ovat **pitkä lyhytkestoinen muisti** (LSTM) ja **kytketty välittäjäyksikkö** (GRU). Käsittelemme tässä LSTM:ää.

![Kuva, joka näyttää esimerkin pitkä lyhytkestoisesta muistisolusta](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM-verkko on järjestetty samankaltaisesti kuin RNN, mutta siinä on kaksi tilaa, jotka kulkevat kerroksesta toiseen: varsinainen tila $c$ ja piilotettu vektori $h$. Jokaisessa yksikössä piilotettu vektori $h_{t-1}$ yhdistetään syötteeseen $x_t$, ja yhdessä ne ohjaavat tilan $c_t$ ja ulostulon $h_t$ käsittelyä **porttien** avulla. Jokaisella portilla on sigmoidi-aktivaatio (lähtöarvo välillä $[0,1]$), jota voidaan ajatella bittikohtaisena maskina, kun sitä kerrotaan tilavektorilla. LSTM:illä on seuraavat portit (kuvassa vasemmalta oikealle):
* **unohtamisportti**, joka määrittää, mitkä vektorin $c_{t-1}$ komponenteista pitää unohtaa ja mitkä päästää läpi.
* **syöteportti**, joka määrittää, kuinka paljon tietoa syötevektorista ja edellisestä piilotetusta vektorista tulisi sisällyttää tilavektoriin.
* **ulostuloportti**, joka ottaa uuden tilavektorin ja päättää, mitkä sen komponenteista käytetään uuden piilotetun vektorin $h_t$ tuottamiseen.

Tilakomponentteja $c$ voi ajatella lipuiksi, joita voidaan kytkeä päälle ja pois. Esimerkiksi, kun kohtamme nimensä *Alice* jaksossa, arvelemme sen viittaavan naiseen ja nostamme lipun tilassa, joka kertoo, että lauseessa on naispuolinen substantiivi. Kun kohtaamme edelleen sanat *and Tom*, nostamme lipun, joka kertoo, että meillä on monikollinen substantiivi. Näin tilaa manipuloimalla pystymme seuraamaan lauseen kielioppisia ominaisuuksia.

> **Huomio**: Tässä on erinomainen resurssi LSTM:ien sisäisten toimintojen ymmärtämiseen: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) kirjoittanut Christopher Olah.

Vaikka LSTM-solun sisäinen rakenne saattaa näyttää monimutkaiselta, Keras piilottaa tämän toteutuksen `LSTM`-kerroksen sisään, joten esimerkissä yllä tarvitsee vain korvata rekurrentti kerros:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Huomaa**, että LSTM:ien kouluttaminen on myös melko hidasta, eikä tarkkuudessa välttämättä näy suurta parannusta koulutuksen alkuvaiheessa. Saatat joutua jatkamaan koulutusta jonkin aikaa saavuttaaksesi hyvän tarkkuuden.


## Kaksisuuntaiset ja monikerroksiset RNN:t

Esimerkeissämme tähän asti rekurrentit verkot toimivat sekvenssin alusta loppuun. Tämä tuntuu luonnolliselta, koska se seuraa samaa suuntaa, johon luemme tai kuuntelemme puhetta. Kuitenkin tilanteissa, joissa syötesekvenssistä tarvitaan satunnaista pääsyä, on järkevämpää suorittaa rekurrentti laskenta molempiin suuntiin. RNN:t, jotka sallivat laskennan molempiin suuntiin, kutsutaan **kaksisuuntaisiksi** RNN:iksi, ja ne voidaan luoda kietomalla rekurrentti kerros erityisellä `Bidirectonal`-kerroksella.

> **Huom**: `Bidirectional`-kerros tekee kaksi kopiota sen sisältämästä kerroksesta ja asettaa yhden kopion `go_backwards`-ominaisuuden arvoksi `True`, jolloin se kulkee sekvenssissä vastakkaiseen suuntaan.

Sekvenssin sisäisiä kuvioita tallentavat rekurrentit verkot, olivatpa ne yksisuuntaisia tai kaksisuuntaisia, tallentavat ne tilavektoreihin tai palauttavat ne ulostulona. Samoin kuin konvoluutioverkoissa, voimme rakentaa toisen rekurrentin kerroksen ensimmäisen jälkeen, jotta siepataan korkeamman tason kuvioita, jotka rakentuvat ensimmäisen kerroksen keräämistä matalamman tason kuvioista. Tämä johdattaa meidät käsitteeseen **monikerroksinen RNN**, joka koostuu kahdesta tai useammasta rekurrentista verkosta, joissa edellisen kerroksen ulostulo siirretään seuraavalle kerrokselle syötteenä.

![Kuva, joka esittää monikerroksista pitkäaikaismuistia hyödyntävää RNN:ää](../../../../../translated_images/fi/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Kuva on peräisin [tästä upeasta kirjoituksesta](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3), kirjoittajana Fernando López.*

Keras tekee näiden verkkojen rakentamisesta helppoa, koska sinun tarvitsee vain lisätä lisää rekurrentteja kerroksia malliin. Kaikille kerroksille paitsi viimeiselle tulee määrittää `return_sequences=True`-parametri, koska tarvitsemme kerroksen palauttamaan kaikki välitilat, ei vain rekurrentin laskennan lopullista tilaa.

Rakennetaan kaksikerroksinen kaksisuuntainen LSTM luokittelutehtäväämme varten.

> **Huom**: tämä koodi kestää taas melko kauan suoritukseltaan, mutta se antaa meille parhaan tarkkuuden, jonka olemme toistaiseksi nähneet. Joten ehkä onkin syytä odottaa ja katsoa tulosta.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN:t muihin tehtäviin

Tähän asti olemme keskittyneet käyttämään RNN:itä tekstijonojen luokitteluun. Mutta ne voivat hoitaa monia muitakin tehtäviä, kuten tekstin generointia ja konekäännöstä — näitä tehtäviä käsittelemme seuraavassa osassa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
